# FAO Crop Price Data Processing

This notebook processes FAO crop price data from downloaded CSV files, combines them, and exports the cleaned data to Stata format.

## Setup and Configuration

Initialize libraries and set up directory paths for input and output files.


In [11]:
# Import libraries
import pandas as pd
import os
import numpy as np
import subprocess # NOTE: Scraper not included in this repo. Please advise @ADAGT
from pathlib import Path

# Base path. Change this to your dropbox folder
mypc_path = "/Users/justinblack/Library/CloudStorage/Dropbox"

# Data and output folder. These should stay the same
data_folder = f'{mypc_path}/Lavoro_Replication/0_data_construction'  
output_folder = f'{data_folder}/0_data_input/data_processed/crop_prices'

# Directory containing the CSV files
source_dir = f'{data_folder}/0_data_input/data_original/crop_prices/fpma_downloads'

# Create directory from output folder and name the file
output_file_name = "fao_crop_prices.csv"
output_path = f"{output_folder}/{output_file_name}"

# For debugging, save locally
# output_path = ""

## Input File Validation

Check for input CSV files and run the scraper if needed. Filter out empty files.


In [12]:
# Get and validate input files
all_files = [os.path.join(source_dir, f)
             for f in os.listdir(source_dir) if f.endswith('.csv')]

# NOTE: Scraper not included in this repo. Please advise @ADAGT
# if not all_files:
#     result = subprocess.run(['python', 'scrape.py'])
#     if result.returncode != 0:
#         print("Error running scrape.py")
#         exit(1)
#     all_files = [os.path.join(source_dir, f)
#                  for f in os.listdir(source_dir) if f.endswith('.csv')]

# Filter out empty files
csv_files = []
for f_path in all_files:
    if os.path.getsize(f_path) > 0:
        csv_files.append(f_path)


## Data Combination

Merge all CSV files into a single DataFrame, ensuring proper date handling and sorting.


In [13]:
# Combine CSV files
combined_df = pd.DataFrame()

# Loop through the CSV files and merge them
for i, file_path in enumerate(csv_files):
    try:
        # Read the current CSV file
        current_df = pd.read_csv(file_path)
        
        # Ensure the 'Date' column exists
        if 'Date' not in current_df.columns:
            print(f"Skipping file {file_path} as it does not have a 'Date' column.")
            continue
        if combined_df.empty:
            combined_df = current_df
        else:
            # Merge based on the 'Date' column
            # We use an outer merge to keep all dates from both dataframes
            combined_df = pd.merge(combined_df, current_df, on='Date', how='outer')
    except pd.errors.EmptyDataError:
        print(f"Skipping empty or malformed file: {file_path}")
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")

# Sort by date if the 'Date' column is present and can be converted to datetime
if 'Date' in combined_df.columns:
    try:
        combined_df['Date'] = pd.to_datetime(combined_df['Date'])
        combined_df = combined_df.sort_values(by='Date')
        # Convert date back to original string format if needed, or keep as datetime
        combined_df['Date'] = combined_df['Date'].dt.strftime('%m/%d/%Y')
    except Exception as e:
        print(f"Could not sort by date: {e}")

print(f"Combined {len(csv_files)} files successfully")

Combined 52 files successfully


## Data Cleaning and Transformation

Clean and transform the data:
1. Format dates to Month/Year format
2. Convert to long format
3. Extract country, location, and crop information
4. Pivot to wide format by crop
5. Add fill-in long and lat coordinates based on market location


In [14]:
# Clean and transform the data
df = combined_df.copy()

# Format the 'Date' column as 'MM, YYYY', and put the result in a new column called 'Month'
df['Month'] = pd.to_datetime(df['Date'], format='%m/%d/%Y').dt.strftime('%m/%Y')

# Drop original Date column to avoid duplicates
df = df.drop('Date', axis=1)

# Melt the DataFrame to long format
df_long = df.melt(id_vars=['Month'], var_name='variable', value_name='price')

def extract_info(s):
    # Example: 'Guinea, RETAIL, Conakry, Beef meat, USD/Kg'
    parts = [x.strip() for x in s.split(',')]
    if len(parts) >= 5:
        country = parts[0]
        location = parts[2]
        crop = parts[3]
        return pd.Series([country, location, crop])
    else:
        return pd.Series([None, None, None])

# Apply extraction
df_long[['Country', 'Location', 'Crop']] = df_long['variable'].apply(extract_info)

# Remove rows with missing info
df_long = df_long.dropna(subset=['Country', 'Location', 'Crop'])

# Pivot so each row is Country, Location, Month, and columns are crops
df_pivot = df_long.pivot_table(index=['Country', 'Location', 'Month'],
                              columns='Crop', values='price', aggfunc='first').reset_index()

# FAO doesn't have latitude and longitude data. Because this data is necessary to map to the shapefile, we can
# hardcode general latitude and longitude coordinates for the marketplaces that appear alongside each price entry 
# in the data. This way, we should be able to get the correct district and chiefdom for the FAO price data. 
# All latitude and longitude coordinates are from https://epsg.io/map. 
# These should cover all of the "Location" entries in the FAO data. 
long_lat_dict = {
    'Saclepea': (6.963811, -8.840873),
    'Bo': (7.962065, -11.736650),
    'Monrovia': (6.328034, -10.797788),
    'Freetown': (8.479004, -13.267950),
    'Kailahun': (8.046631, -10.656579),
    'Kenema': (7.885973, -11.186383),
    'Koinadugu': (9.330514, -11.354295),
    'Conakry': (9.592818, -13.657054),
    'Faranah': (10.407280, -10.149943),
    'N\'Zérékoré': (8.389897, -9.169890),
}   

# Add latitude and longitude columns
df_pivot['Latitude'] = df_pivot['Location'].map(lambda x: long_lat_dict[x][0] if x in long_lat_dict else np.nan)
df_pivot['Longitude'] = df_pivot['Location'].map(lambda x: long_lat_dict[x][1] if x in long_lat_dict else np.nan)

# Rename columns to be Stata-compliant to avoid warnings
column_rename_dict = {
    'Beef meat': 'Beef_meat',
    'Cassava (fresh)': 'Cassava__fresh_',
    'Fish (Bonga': 'Fish__Bonga',
    'Groundnuts (shelled)': 'Groundnuts__shelled_',
    'Palm Oil': 'Palm_Oil',
    'Palm Oil (traditional)': 'Palm_Oil__traditional_',
    'Rice (imported': 'Rice__imported',
    'Rice (imported)': 'Rice__imported_',
    'Rice (parboiled': 'Rice__parboiled',
    'Wheat (flour)': 'Wheat__flour_'
}

# Apply column renaming only for columns that exist in the dataframe
df_pivot.rename(columns={k: v for k, v in column_rename_dict.items() if k in df_pivot.columns}, inplace=True)

## Export to .csv

Export to .csv format:
1. Convert numeric columns to proper float format
2. Handle string columns appropriately

In [15]:
# Prepare and export to Stata format

# Convert relevant numeric columns to float for Stata compatibility
numeric_columns = df_pivot.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_columns:
    df_pivot[col] = pd.to_numeric(df_pivot[col], errors='coerce')

# Convert string columns to proper format for Stata
for col in df_pivot.select_dtypes(include='object').columns:
    unique_types = set(type(x) for x in df_pivot[col].dropna())
    if all(t in [str, type(None)] for t in unique_types):
        df_pivot[col] = df_pivot[col].astype(str)
    elif df_pivot[col].isnull().all():
        df_pivot.drop(columns=[col], inplace=True)
    else:
        print(f"⚠️ Column '{col}' has unsupported types: {unique_types}")
        df_pivot.drop(columns=[col], inplace=True)

# Export to Stata format
df_pivot.to_csv(output_path, index=False)

print(f"\nData exported at: {output_path}")


Data exported at: /Users/justinblack/Library/CloudStorage/Dropbox/Lavoro_Replication/0_data_construction/0_data_input/data_processed/crop_prices/fao_crop_prices.csv
